# Forecasting Financial Statements with `finstmt`

First import the necessary classes and `pandas`.

In [ ]:
from pathlib import Path

import pandas as pd

from finstmt import FinancialStatements
from finstmt.config.statement import STATEMENT_CONFIGS

## Load Financial Statements

The sample data is Caterpillar's annual income statement and balance sheet. `STATEMENT_CONFIGS` holds finstmt's bundled income statement, balance sheet and metrics configs.

In [ ]:
data_folder = Path("..") / "tests" / "sources" / "stockrow" / "CAT"
inc_df = pd.read_csv(data_folder / "annual_income.csv", index_col=0)
bs_df = pd.read_csv(data_folder / "annual_bs.csv", index_col=0)

stmts = FinancialStatements.from_df(
    pd.concat([inc_df, bs_df]), STATEMENT_CONFIGS, disp_unextracted=False
)

### View Existing Statements

In [ ]:
stmts

### Copy the Statements

To run several forecasts from the same data, copy the statements with `stmts.copy()`. Later configuration changes then affect only the original, not the copy.

In [ ]:
stmts2 = stmts.copy()

## Run a Forecast

### Set the Default Method

This step is optional. It shows how to change the default forecast method; the default is already `cagr`, so it changes nothing here.

In [ ]:
stmts.config.update_all(["forecast", "method"], "cagr")

### Keep calculated items calculated

finstmt switches a calculated item to forecasting when the inputs to its formula are missing from the data. In this data that happens to cash and short-term investments, net PP&E and operating expenses. Forecasting cash and short-term investments directly would cut the link between the cash plug and total assets, so the balance sheet could not balance. Switch them back to calculated:

In [ ]:
for key in ["cash_and_st_invest", "net_ppe", "op_exp"]:
    stmts.config.update(key, ["forecast", "make_forecast"], False)

### View the Forecast Assumptions

All the assumptions going into the forecast are in `forecast_assumptions`:

In [ ]:
stmts.forecast_assumptions

### Run the First Forecast

`.forecast()` runs a forecast and returns `ForecastedStatements`. Run a forecast first, look at the results, then adjust.

In [ ]:
fcst = stmts.forecast()

> Note: CAGR warns about items that start or end at zero, or change sign, and forecasts zero growth for them.

Now view the forecast.

In [ ]:
fcst

And the plot:

In [ ]:
fcst.plot()

You can also plot a subset. Focus on a few items:

In [ ]:
fcst.plot(subset=["revenue", "cogs", "cash"])

### Change Methods and Re-Run the Forecast

After viewing the plots, choose the forecast method for each item:

In [ ]:
stmts.config.update("revenue", ["forecast", "method"], "trend")
stmts.config.update("cogs", ["forecast", "method"], "mean")

Now re-run the forecast and plot the results:

In [ ]:
fcst = stmts.forecast()
fcst.plot(subset=["revenue", "cogs", "cash"])

### Adjust an Existing Forecast

It is often easiest to run a forecast and then adjust it. `.to_manual` turns an item's forecast into a manual one, applying `adjustments` or `replacements` to its growth rates or, with `use_levels=True`, to its levels. Pass a dictionary keyed by forecast period index, or a list with a value for every period.

In [ ]:
fcst.forecasts["revenue"].to_manual(adjustments={0: 0.05})  # 5 points more growth in the first period

COGS is forecast as a percent of revenue, which `to_manual` does not support yet. Set its ratio in the configuration instead:

In [ ]:
stmts.config.update("cogs", ["forecast", "method"], "manual")
stmts.config.update("cogs", ["forecast", "manual_forecasts"], {"levels": [0.65] * 5, "growth": []})

Run the forecast again to update the statements, and plot them:

In [ ]:
fcst = stmts.forecast()
fcst.plot(subset=["revenue", "cogs", "cash"])

## A Second Forecast

The copy made earlier still has the original assumptions, so it can be used for a separate forecast:

In [ ]:
stmts2.forecast_assumptions